# JGE Part 4 — Coupled prospectivity and petrogenetic attribution

This notebook consumes saved OOF outputs from Task A and Task B. It does not refit either model. The figure-audited v2 workflow performs dependency-block inference, applies the predeclared readiness gates, and exports Figures 6–9 plus Figures S1–S6 with panel-level source data.

**Interpretation boundary:** SHAP is model attribution, not geological causality. The complete I/A/S linkage is not promoted when the Task B overall readiness contract is not met.

In [ ]:
from pathlib import Path
import json
import platform
import sys

import matplotlib
import numpy as np
import pandas as pd
import scipy

cwd = Path.cwd().resolve()
if cwd.name == 'notebooks':
    PROJECT_ROOT = cwd.parent
elif (cwd / 'notebooks').is_dir():
    PROJECT_ROOT = cwd
else:
    raise RuntimeError('Open this notebook from the coupling workflow folder or its notebooks directory.')

SOURCE_DIR = PROJECT_ROOT / 'src'
CONFIG_PATH = PROJECT_ROOT / 'config' / 'coupling_config.json'
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

import importlib
import coupling_pipeline
coupling_pipeline = importlib.reload(coupling_pipeline)
run_pipeline = coupling_pipeline.run_pipeline

print('Project root:', PROJECT_ROOT)
print('Python:', platform.python_version())
print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('scipy:', scipy.__version__)
print('matplotlib:', matplotlib.__version__)

## Run the complete guarded workflow

The formal configuration uses 2,000 dependency-block bootstrap replicates and 9,999 structure-preserving permutations. Figure 8 is automatically routed to Main_Text or Supplementary according to the gate. Supplementary Figures S1–S6 are always generated.

In [ ]:
result = run_pipeline(CONFIG_PATH, make_figures=True)
print(json.dumps(result['decision'], ensure_ascii=False, indent=2))

## Cohort and readiness audit

In [ ]:
display(result['cohort_flow'])
display(result['model_readiness'])

## Cross-task attribution and conditional associations

Raw SHAP magnitudes are not compared across models. The table uses feature direction, within-task contribution share and locked-feature correspondence.

In [ ]:
display(result['feature_concordance'])
display(result['association_results'])

## Output files

In [ ]:
print('Figure placement decision:', result['decision']['figure8_placement'])
for path in result['figure_files']:
    print(path)

output_root = Path(result['manifest']['output_root'])
print('\nRun summary:', output_root / '04_Logs' / 'RUN_SUMMARY.md')
print('Run manifest:', output_root / '04_Logs' / 'run_manifest.json')
print('Figure contracts:', output_root / '04_Logs' / 'figure_contract_manifest.csv')
main_files = list((output_root / '03_Figures' / 'Main_Text').glob('*'))
supp_files = list((output_root / '03_Figures' / 'Supplementary').glob('*'))
print(f'Main-text files: {len(main_files)}; Supplementary files: {len(supp_files)}')
assert len(supp_files) >= 6 * 4, 'Expected at least Figures S1-S6 in SVG/PDF/TIFF/PNG.'

## Manuscript claim check

- If restricted_s_diagnostic_eligible is true but the raw probability association gate is false, retain Figure 7 as an attribution correspondence and move Figure 8 to the Supplementary Material.
- Do not interpret weak I/A coupling as evidence that I/A granites are geologically unrelated to uranium mineralization.
- Do not claim improved prediction unless a separate nested incremental-prediction comparison is completed.